# 04 · Quality score calibration: NYC Yellow Taxi

Phase 4, Step 4.7. Shows the quality score per month, which categories cost points, how much
the score depends on the weights in `configs/severity_weights.yml`, and the incident timeline.

**Before running:**
```
python src/ingestion/ingest_data.py
python src/silver/process_silver.py
python src/quality/run_checks.py
python src/quality/run_rules.py
python src/incidents/incidents_cli.py update
python src/quality/score.py
```
The last section prints a verdict to paste back.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

from common import db
from common.config import load_datasets_config, load_severity_weights
from ingestion.bronze_loader import current_versions
from quality.score import Part, combine

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)

## Settings

In [ ]:
config = load_datasets_config()
dataset = config.dataset("yellow_taxi")
DATABASE = Path(os.environ.get("DATAGUARD_DATABASE", config.paths["duckdb"]))
weights_config = load_severity_weights()
weights = weights_config["category_weights"]
print("weights:", weights)
print("check scores:", weights_config["check_scores"])

con = db.connect(DATABASE, read_only=True)
current = pd.DataFrame(
    [(p, sha) for p, (_, sha) in current_versions(con, dataset).items()], columns=["period", "sha256"]
)
scores = con.execute(
    "SELECT * FROM quality_scores WHERE dataset = ? ORDER BY period", [dataset.name]
).df().merge(current, on=["period", "sha256"])
parts = con.execute(
    "SELECT * FROM quality_score_parts WHERE dataset = ?", [dataset.name]
).df().merge(current, on=["period", "sha256"])
assert not scores.empty, "no scores found: run python src/quality/score.py first"
print(f"{len(scores)} month(s) scored")

## 1 · Score per month

In [ ]:
scores[["period", "score", "rows", "check_run"]].round({"score": 1})

In [ ]:
fig = px.bar(scores, x="period", y="score", range_y=[0, 100], title="Quality score per month")
fig.show()

## 2 · Pass rate per category

Rows are months, columns are categories (in %). An empty cell means the category had no result
that month and was left out of the score.

In [ ]:
rates = parts.pivot(index="period", columns="category", values="pass_rate")
(100 * rates).round(2)

## 3 · Points lost per category

`weight × (1 − pass rate) / Σ weights × 100`: how many of the 100 points each category cost.
The columns of a month add up to `100 − score`.

In [ ]:
totals = parts.groupby("period")["weight"].transform("sum")
parts["points_lost"] = 100 * parts["weight"] * (1 - parts["pass_rate"]) / totals
lost = parts.pivot(index="period", columns="category", values="points_lost").fillna(0)
lost.round(2)

In [ ]:
fig = px.bar(parts, x="period", y="points_lost", color="category",
             title="Points lost per category")
fig.show()

In [ ]:
parts[parts["points_lost"] > 0.5].sort_values(["period", "points_lost"], ascending=[True, False])[
    ["period", "category", "pass_rate", "points_lost", "source"]
]

## 4 · Weight sensitivity

The same pass rates scored with the configured weights and with all weights equal. If the
months keep their order, the score does not hinge on the exact weights.

In [ ]:
def rescore(w):
    return {
        period: combine([Part(r.category, w[r.category], r.pass_rate, "") for r in g.itertuples()])
        for period, g in parts.groupby("period")
    }

equal = {c: 1.0 for c in weights}
sensitivity = pd.DataFrame({"configured": rescore(weights), "equal_weights": rescore(equal)})
sensitivity["rank_configured"] = sensitivity["configured"].rank(method="min").astype(int)
sensitivity["rank_equal"] = sensitivity["equal_weights"].rank(method="min").astype(int)
sensitivity.round(1)

## 5 · Incidents

In [ ]:
incidents = con.execute(
    "SELECT * FROM incidents WHERE dataset = ? ORDER BY incident_id", [dataset.name]
).df()
print(f"{len(incidents)} incident(s), {int((incidents['status'] != 'RESOLVED').sum())} active")
incidents[["incident_id", "source", "rule_id", "column_name", "severity", "status",
           "first_seen_period", "last_seen_period", "occurrences", "resolved_period",
           "recurrence_of"]]

Timeline: each bar runs from the first month an incident was seen to the month it was
resolved (or the last month it was seen, while it is active).

In [ ]:
if not incidents.empty:
    timeline = incidents.copy()
    timeline["start"] = pd.to_datetime(timeline["first_seen_period"] + "-01")
    end = timeline["resolved_period"].fillna(timeline["last_seen_period"])
    timeline["end"] = pd.to_datetime(end + "-01") + pd.offsets.MonthEnd(0)
    timeline["label"] = timeline["incident_id"] + " " + timeline["rule_id"] + " " + timeline["column_name"]
    fig = px.timeline(timeline, x_start="start", x_end="end", y="label", color="severity",
                      title="Incident timeline")
    fig.update_yaxes(autorange="reversed")
    fig.show()

In [ ]:
opened = incidents.groupby("first_seen_period").size().rename("incidents_opened")
scores.set_index("period")[["score"]].join(opened).fillna({"incidents_opened": 0}).astype({"incidents_opened": int}).round(1)

## 6 · Calibration verdict

Copy the printed text back.

In [ ]:
top_loss = lost.idxmax(axis=1).where(lost.max(axis=1) > 0)
verdict = {
    "months": len(scores),
    "scores": scores.set_index("period")["score"].round(1).to_dict(),
    "lowest": scores.nsmallest(3, "score")["period"].tolist(),
    "main_loss": top_loss.dropna().to_dict(),
    "same_order_with_equal_weights": bool(
        (sensitivity["rank_configured"] == sensitivity["rank_equal"]).all()
    ),
    "incidents": incidents.groupby("severity").size().to_dict(),
    "active_incidents": int((incidents["status"] != "RESOLVED").sum()),
}
print(json.dumps(verdict, indent=2, default=str))

In [ ]:
con.close()